In [5]:
from pyspark.sql import functions as F

# Read cleaned Taxi data from Silver Lakehouse
taxi_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.yellow_taxi_clean"
)

print("Silver Taxi records:", taxi_df.count())

# Build daily Taxi fact table
gold_taxi_df = (
    taxi_df
    .groupBy("pickup_date", "puLocationID")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("passenger_count").alias("total_passengers"),
        F.sum("trip_distance").alias("total_distance"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("total_amount").alias("avg_fare")
    )
    .withColumnRenamed("pickup_date", "date")
    .withColumnRenamed("puLocationID", "pickup_location_id")
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("pickup_hour", F.lit(None).cast("int"))
)

# Save to Gold Lakehouse
(
    gold_taxi_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("FactTaxiDaily")
)

print("Gold Taxi table saved successfully!")
print("Gold records:", gold_taxi_df.count())

display(gold_taxi_df.orderBy("date", "pickup_location_id"))

StatementMeta(, e04c8bd4-2823-4407-9e3f-58fcce4aee6b, 6, Finished, Available, Finished, False)

Silver Taxi records: 3555245
Gold Taxi table saved successfully!
Gold records: 7519


SynapseWidget(Synapse.DataFrame, 7802db2d-d3d8-4771-bf10-e4e456d239d1)

In [6]:
from pyspark.sql import functions as F

taxi_dates = spark.table("FactTaxiDaily").select("date")
air_dates = spark.table("FactAirQualityDaily").select("date")

all_dates = (
    taxi_dates
    .union(air_dates)
    .distinct()
    .filter(F.col("date").isNotNull())
)

dim_date = (
    all_dates
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .orderBy("date")
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimDate")
)

print("DimDate created successfully!")
print("Total dates:", dim_date.count())

display(dim_date)

StatementMeta(, e04c8bd4-2823-4407-9e3f-58fcce4aee6b, 7, Finished, Available, Finished, False)

DimDate created successfully!
Total dates: 33


SynapseWidget(Synapse.DataFrame, 90af64d7-403b-45a7-a09d-06348aee8f06)

In [7]:
from pyspark.sql import functions as F

taxi_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Silver_LH.dbo.yellow_taxi_clean"
)

dim_zone = (
    taxi_df
    .select(F.col("pulocationid").alias("location_id"))
    .filter(F.col("location_id").isNotNull())
    .distinct()
    .withColumn("zone_key", F.col("location_id").cast("int"))
    .withColumn("borough", F.lit("Unknown"))
    .withColumn("zone_name", F.concat(F.lit("Location "), F.col("location_id")))
    .select("zone_key", "location_id", "borough", "zone_name")
    .orderBy("zone_key")
)

(
    dim_zone.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimZone")
)

print("DimZone created successfully!")
print("Total zones:", dim_zone.count())

display(dim_zone)

StatementMeta(, e04c8bd4-2823-4407-9e3f-58fcce4aee6b, 8, Finished, Available, Finished, False)

DimZone created successfully!
Total zones: 262


SynapseWidget(Synapse.DataFrame, 687bda53-c74b-4ec0-93b6-488b270cb97b)